# 07 — Phase 1: Message-Passing Aggregation Ablation

Staged ablation study, Phase 1 of 4 (aggregation → query features → protein-size
weighting → gradient accumulation). This phase isolates the `MessageLayer`
aggregation choice — `mean`, `sum`, `max`, or `multi` (mean+sum+max concatenated)
— for both `DistanceESPN` and `AttentionESPN`, with query node features held off
and batching held at defaults so the comparison isn't confounded by anything else.
See `sweeps/phase1_aggregation_ablation.yaml` for the run definitions.

## Selection methodology

**Winners are chosen from validation metrics (Section 4), not test metrics
(Section 5).** `07_train.py` auto-evaluates every run against the test set on
completion — using those numbers to pick a phase winner would mean implicitly
fitting the test set across all 4 rounds of ablation selection. Section 5 is
included only as a final sanity check, not as the basis for the decision.

Primary metric: per-run validation Pearson r, taken from the epoch that
produced `best_model.pt` (i.e. the epoch with minimum val loss, read from
`metrics.csv`). Secondary/tie-break: validation RMSE. Also watch the
train/val gap — a config with slightly lower val Pearson r but a much
smaller train/val gap generalizes better and may be the better pick despite
not topping the leaderboard.

Aggregation for `AttentionESPN` only affects the bond/radial/qq
(`_AtomMP`/`_QueryRefine`) stages — its AQ stage is always cross-attention
regardless of this setting.

## Prerequisites

- All 8 runs in `sweeps/phase1_aggregation_ablation.yaml` completed
  (`run_sweep.py sweeps/phase1_aggregation_ablation.yaml --all`)
- `scripts/analyze_model.py --curves --distributions --save-plots` has been run
  for each checkpoint dir, to produce the PNGs Sections 2-3 load

## Decision

*To be filled in once Phase 1 results exist — see Section 7.*

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path("../..").resolve()))


def show_png_grid(runs, filename, title, ncols=2):
    """Display saved PNG plots from each run's plot_dir in a grid."""
    available = [r for r in runs if (r["plot_dir"] / filename).exists()]
    if not available:
        print(f"No '{filename}' plots found. Run analyze_model.py --save-plots first.")
        return
    nrows = (len(available) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows))
    axes = axes.flatten() if nrows * ncols > 1 else [axes]
    fig.suptitle(title, fontsize=14, fontweight="bold", y=1.01)
    for ax, run in zip(axes, available):
        ax.imshow(mpimg.imread(run["plot_dir"] / filename))
        ax.set_title(run["label"], fontsize=11)
        ax.axis("off")
    for ax in axes[len(available):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()


def plot_metric_bars(df, title_prefix, metrics):
    """1-row x 2-col grouped bar chart: one chart per model type.

    metrics: list of (column_name, color) tuples — e.g. validation runs have
    no MAE (metrics.csv doesn't track it), test runs do.
    """
    if df.empty:
        print("No data to plot.")
        return
    model_types = ["Attention", "Distance"]
    fig, axes = plt.subplots(1, len(model_types), figsize=(14, 5), sharey=False)
    fig.suptitle(title_prefix, fontsize=13, fontweight="bold")
    n_metrics = len(metrics)
    total_width = 0.7
    bar_w = total_width / n_metrics
    for ax, model_type in zip(axes, model_types):
        sub = df[df["Model"] == model_type].copy()
        if sub.empty:
            ax.set_visible(False)
            continue
        x = range(len(sub))
        for i, (metric, color) in enumerate(metrics):
            if metric not in sub.columns or sub[metric].isna().all():
                continue
            offsets = [xi - total_width / 2 + bar_w * i + bar_w / 2 for xi in x]
            bars = ax.bar(offsets, sub[metric], width=bar_w, color=color, label=metric, zorder=3)
            ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=8, rotation=90)
        ax.set_title(model_type, fontsize=12)
        ax.set_xlabel("Aggregation")
        ax.set_xticks(list(x))
        ax.set_xticklabels(sub["Aggregation"].tolist(), fontsize=10)
        ax.legend(fontsize=9)
        ax.grid(axis="y", alpha=0.3, zorder=0)
    plt.tight_layout()
    plt.show()

## 1. Configuration

`CKPT_ROOT` and `EVAL_ROOT` point to the checkpoint and evaluation output
directories where all 8 Phase 1 runs have been trained and evaluated
(`07_train.py`'s default checkpoint dir is `<data_root>/../checkpoints/<model>_<suffix>`
— edit `THESIS_ROOT` below to match wherever you're running this, e.g. copied
back from the research node). The `RUNS` list mirrors
`sweeps/phase1_aggregation_ablation.yaml` exactly — suffix values there are
just `mean`/`sum`/`max`/`multi` (no model-name duplication).

In [ ]:
THESIS_ROOT = Path("/home/student/thesis")
CKPT_ROOT   = THESIS_ROOT / "checkpoints"
EVAL_ROOT   = THESIS_ROOT / "model_eval"

AGGS = ["mean", "sum", "max", "multi"]

RUNS = [
    dict(label=f"{model.capitalize()} — {agg}", model_type=model, agg=agg,
         plot_dir=EVAL_ROOT/f"{model}_{agg}", ckpt_dir=CKPT_ROOT/f"{model}_{agg}")
    for model in ["attention", "distance"]
    for agg in AGGS
]

print(f"{'Run':<22}  {'Plots':>6}  {'Metrics':>8}")
print("-" * 42)
for r in RUNS:
    has_plots   = r["plot_dir"].exists()
    has_metrics = (r["ckpt_dir"] / "metrics.csv").exists()
    print(f"{r['label']:<22}  {'yes' if has_plots else 'no':>6}  {'yes' if has_metrics else 'no':>8}")

## 2. Training Curves

Plots saved by `scripts/analyze_model.py --curves` for each run.

In [ ]:
show_png_grid(RUNS, "training_curves.png", "Training Curves — Aggregation Ablation")

## 3. Error Distributions

Plots saved by `scripts/analyze_model.py --distributions` for each run.

In [ ]:
show_png_grid(RUNS, "error_distributions.png", "Error Distributions — Aggregation Ablation", ncols=1)

## 4. Validation Metrics Comparison (selection basis)

Loads `metrics.csv` from each checkpoint dir (written every epoch by
`Trainer._save_metrics_csv`) and takes the row at the epoch with minimum
`val_loss` — i.e. the epoch `best_model.pt` was actually saved from. This is
what the Phase 1 winner (per architecture) should be chosen from, not
Section 5's test metrics.

**Train/val gap** is `train_loss` minus `val_loss` at that same epoch — a
larger gap suggests more overfitting to the training subsample; prefer the
smaller-gap config when validation scores are close.

In [ ]:
rows = []
for run in RUNS:
    csv_path = run["ckpt_dir"] / "metrics.csv"
    if not csv_path.exists():
        continue
    hist = pd.read_csv(csv_path)
    if hist.empty:
        continue
    best = hist.loc[hist["val_loss"].idxmin()]
    rows.append({
        "Run":          run["label"],
        "Model":        run["model_type"].capitalize(),
        "Aggregation":  run["agg"],
        "Pearson r":    best["val_pearson_r"],
        "RMSE":         best["val_rmse"],
        "Val loss":     best["val_loss"],
        "Train loss":   best["train_loss"],
        "Train/val gap": best["train_loss"] - best["val_loss"],
        "Best epoch":   int(best["epoch"]),
    })

val_df = pd.DataFrame(rows).sort_values(["Model", "Pearson r"], ascending=[True, False])
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
display(val_df)

In [ ]:
plot_metric_bars(val_df, "Validation metrics — Aggregation ablation (selection basis)",
                  metrics=[("Pearson r", "steelblue"), ("RMSE", "darkorange")])

## 5. Test Metrics (reference only — not used for phase selection)

Same shape as Section 4 but from `test_metrics.json`. Useful as a sanity
check — if the test ranking disagrees sharply with the validation ranking
from Section 4, that's worth investigating before locking in a winner — but
the winner itself should come from Section 4.

In [ ]:
rows_test = []
for run in RUNS:
    metrics_path = run["ckpt_dir"] / "test_metrics.json"
    if not metrics_path.exists():
        continue
    with open(metrics_path) as f:
        data = json.load(f)
    g = data.get("global", {})
    rows_test.append({
        "Run":            run["label"],
        "Model":          run["model_type"].capitalize(),
        "Aggregation":    run["agg"],
        "Pearson r":      g.get("pearson_r"),
        "RMSE":           g.get("rmse"),
        "MAE":            g.get("mae"),
        "Train time (s)": g.get("train_wall_time_s"),
        "N proteins":     g.get("n_proteins"),
    })

test_df = pd.DataFrame(rows_test).sort_values(["Model", "Pearson r"], ascending=[True, False])
display(test_df)

In [ ]:
plot_metric_bars(test_df, "Test metrics — Aggregation ablation (reference only)",
                  metrics=[("Pearson r", "steelblue"), ("RMSE", "darkorange"), ("MAE", "seagreen")])

## 6. Per-protein Consistency

Per-protein Pearson r (from `test_metrics.json`, purely descriptive here —
not a selection input) across all 8 runs. Same diagnostic purpose as in the
original feature-ablation notebook: which proteins are universally easy or
hard, and which are aggregation-sensitive (high rank variance across the 4
agg choices)?

In [ ]:
import numpy as np

pp_data: dict[str, dict[str, float]] = {}

for run in RUNS:
    metrics_path = run["ckpt_dir"] / "test_metrics.json"
    if not metrics_path.exists():
        continue
    with open(metrics_path) as f:
        data = json.load(f)
    pp = data.get("per_protein", {})
    pp_data[run["label"]] = {pid: v["pearson_r"] for pid, v in pp.items()}

all_proteins = sorted({pid for d in pp_data.values() for pid in d})
run_labels   = list(pp_data.keys())

pivot = pd.DataFrame(
    {label: [pp_data[label].get(pid, float("nan")) for pid in all_proteins]
     for label in run_labels},
    index=all_proteins,
)

def _short(pid):
    return pid.removeprefix("AF-").removesuffix("-F1")

pivot.index = [_short(p) for p in pivot.index]
pivot.columns = [
    label.replace("Attention — ", "Attn/").replace("Distance — ", "Dist/")
    for label in pivot.columns
]

pivot["_mean"] = pivot.mean(axis=1)
pivot = pivot.sort_values("_mean", ascending=False).drop(columns=["_mean"])

print(f"Loaded per-protein data for {len(pivot)} proteins across {len(run_labels)} runs.")

In [ ]:
n_prot, n_runs = pivot.shape
fig, ax = plt.subplots(figsize=(max(8, n_runs * 1.2), max(6, n_prot * 0.38 + 1)))

mat = pivot.values.astype(float)
vmin = np.nanmin(mat) - 0.01
vmax = np.nanmax(mat) + 0.01

im = ax.imshow(mat, aspect="auto", cmap="RdYlGn", vmin=vmin, vmax=vmax)

ax.set_xticks(range(n_runs))
ax.set_xticklabels(pivot.columns, fontsize=9, rotation=30, ha="right")
ax.set_yticks(range(n_prot))
ax.set_yticklabels(pivot.index, fontsize=8)

for i in range(n_prot):
    for j in range(n_runs):
        v = mat[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5,
                    color="black" if 0.3 < v < 0.85 else "white")

row_means = np.nanmean(mat, axis=1)
worst_idx = int(np.argmin(row_means))
best_idx  = int(np.argmax(row_means))
for idx, color in [(worst_idx, "#d62728"), (best_idx, "#2ca02c")]:
    ax.add_patch(plt.Rectangle((-0.5, idx - 0.5), n_runs, 1,
                               linewidth=2, edgecolor=color, facecolor="none"))

plt.colorbar(im, ax=ax, label="Pearson r", shrink=0.6)
ax.set_title("Per-protein Pearson r across all 8 aggregation runs\n"
             "(green outline = best protein, red = worst; sorted by mean r)",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
rank_df = pivot.rank(ascending=False, method="min").astype(int)

rank_df.insert(0, "Mean r",    pivot.mean(axis=1).round(4))
rank_df.insert(1, "Std r",     pivot.std(axis=1).round(4))
rank_df.insert(2, "Mean rank", rank_df.iloc[:, 2:].mean(axis=1).round(2))
rank_df.insert(3, "Rank std",  rank_df.iloc[:, 3:].std(axis=1).round(2))

rank_df = rank_df.sort_values("Mean rank")

print("Top 5 proteins (lowest mean rank = most consistently well-predicted):")
display(rank_df.head(5))

print("\nBottom 5 proteins (highest mean rank = most consistently difficult):")
display(rank_df.tail(5))

print("\nMost aggregation-sensitive proteins (highest rank std):")
display(rank_df.sort_values("Rank std", ascending=False).head(5))

## 7. Error Distribution by ESP Value

Does the model struggle more at extreme positive/negative ESP values, and
does that vary by protein net charge? (SUMMER_PLAN.md "Error Distribution by
ESP Value".) Plots saved by `scripts/analyze_model.py --error-by-esp` for
each run.

Left panel: residual (pred − true) vs ground-truth ESP, coloured by net
charge, with a binned median |residual| trend line — a V-shape or upward
trend at the edges means error grows with ESP magnitude. Right panel:
residual histograms split by |ESP| tertile — a wider "high |ESP|"
distribution means the model is systematically less accurate on extreme
values, not just noisier on them.

In [ ]:
show_png_grid(RUNS, "error_by_esp.png", "Error Distribution by ESP Value — Aggregation Ablation", ncols=1)

## 8. Decision

*Fill in once Phase 1 completes. Template:*

| Model | Winning aggregation | Val Pearson r | Val RMSE | Train/val gap | Runner-up |
|---|---|---|---|---|---|
| Attention | ? | ? | ? | ? | ? |
| Distance | ? | ? | ? | ? | ? |

**Reasoning:** *why the winner was chosen over the runner-up — especially if
it wasn't simply the top row of Section 4 (e.g. picked for a smaller
train/val gap, or because the margin over the runner-up was within noise
for this subsample size).*

**Carried forward to Phase 2:** *the winning `agg` value for each
architecture, locked in for the query-feature ablation sweep (notebook 08).*

**3D visualization / deeper per-protein diagnostics** are deferred to the
final cross-phase comparison (after Phase 4) rather than repeated at every
stage — see `notebooks/decisions/14_embedding_analysis.ipynb`,
`15_partial_charge_probe.ipynb`, and `16_coulomb_baseline.ipynb`.